# Phase 7 — Calibration: make the confidence honest

The trained network gives a confidence with every answer ("92% Speckled"). **Calibration** asks whether
that number can be trusted: of all the cells where it says about 90%, are about 90% really correct?

This notebook trains nothing. It reads the saved outputs of Phase 5 (`image_logits.csv` of every round)
and needs **no GPU**; it finishes in under a minute.

**What it does, per round**
1. Fits one number, the **temperature** *T*, on the **validation** images.
2. Divides the network's raw outputs by *T* and measures the effect on the **test** images.

*T* above 1 makes the network less sure of itself; below 1 makes it more sure. Dividing by *T* never
changes which class wins, so accuracy stays exactly the same. Only the confidence changes.

In [ ]:
# ============================== 1. SETTINGS ==============================
RUN_NAME = "resnet18_adjusted"           # the run chosen in Phase 5
ROUNDS = [0, 1, 2, 3, 4]
CLASS_NAMES = ["Homogeneous", "Speckled", "Nucleolar"]
N_BINS = 15                              # confidence is cut into 15 equal ranges for the calibration error
THRESHOLDS = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]   # "only accept answers at least this confident"

In [ ]:
# COLAB-ONLY ============================== 2. DRIVE ==============================
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
OUT_ROOT = Path("/content/drive/MyDrive/ANA_Pattern3/outputs")

In [ ]:
# ============================== 3. LOAD THE SAVED OUTPUTS ==============================
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.special import log_softmax

RUN_DIR = Path(OUT_ROOT) / RUN_NAME
CAL_DIR = RUN_DIR / "calibration"; CAL_DIR.mkdir(exist_ok=True)
LOGIT_COLUMNS = [f"logit_{c}" for c in CLASS_NAMES]

data = {}                                # round -> {"val": (logits, labels), "test": (logits, labels)}
for k in ROUNDS:
    frame = pd.read_csv(RUN_DIR / f"round_{k}" / "image_logits.csv")
    assert set(frame["split"]) == {"val", "test"}, f"round {k}: unexpected split names"
    val_files = set(frame.loc[frame["split"] == "val", "file_index"])
    test_files = set(frame.loc[frame["split"] == "test", "file_index"])
    assert not (val_files & test_files), f"round {k}: an image is in both validation and test"
    data[k] = {name: (part[LOGIT_COLUMNS].to_numpy(np.float64), part["label"].to_numpy())
               for name, part in frame.groupby("split")}
    print(f"round {k}: {len(val_files)} validation images, {len(test_files)} test images")

In [ ]:
# ============================== 4. THE MEASUREMENTS ==============================
def probabilities(logits, T=1.0):
    return np.exp(log_softmax(logits / T, axis=1))          # softmax: turns raw outputs into shares that add up to 1


def nll(logits, y, T=1.0):
    # Negative log-likelihood (the training loss). Punishes confident wrong answers hard. Lower is better.
    return -log_softmax(logits / T, axis=1)[np.arange(len(y)), y].mean()


def brier(logits, y, T=1.0):
    # Mean squared distance between the predicted shares and the true answer (0 = perfect).
    p = probabilities(logits, T)
    return ((p - np.eye(p.shape[1])[y]) ** 2).sum(axis=1).mean()


def bins(logits, y, T=1.0):
    # Group the images by confidence; per group: how many, mean confidence, share that is correct.
    p = probabilities(logits, T)
    confidence, correct = p.max(axis=1), p.argmax(axis=1) == y
    which = np.minimum((confidence * N_BINS).astype(int), N_BINS - 1)
    rows = [(int((which == b).sum()), confidence[which == b].mean(), correct[which == b].mean())
            for b in range(N_BINS) if (which == b).any()]
    return pd.DataFrame(rows, columns=["n", "confidence", "accuracy"])


def ece(logits, y, T=1.0):
    # Expected calibration error: the average gap between confidence and accuracy, weighted by group size.
    table = bins(logits, y, T)
    return float((table["n"] / table["n"].sum() * (table["confidence"] - table["accuracy"]).abs()).sum())


def fit_temperature(logits, y):
    # The T that gives the lowest loss on these images. Searched on a log scale between 0.05 and 20.
    best = minimize_scalar(lambda log_t: nll(logits, y, np.exp(log_t)), bounds=(np.log(0.05), np.log(20)),
                           method="bounded")
    return float(np.exp(best.x))

In [ ]:
# ============================== 5. FIT ON VALIDATION, MEASURE ON TEST ==============================
rows, temperatures = [], {}
for k in ROUNDS:
    (val_logits, val_y), (test_logits, test_y) = data[k]["val"], data[k]["test"]
    T = fit_temperature(val_logits, val_y)                   # validation only: the test fold is not touched here
    temperatures[k] = T
    p = probabilities(test_logits)
    assert (p.argmax(1) == probabilities(test_logits, T).argmax(1)).all()   # T never changes the answer
    rows.append({"round": k, "temperature": T,
                 "test_accuracy": (p.argmax(1) == test_y).mean(),
                 "mean_conf_before": p.max(1).mean(), "mean_conf_after": probabilities(test_logits, T).max(1).mean(),
                 "ece_before": ece(test_logits, test_y), "ece_after": ece(test_logits, test_y, T),
                 "nll_before": nll(test_logits, test_y), "nll_after": nll(test_logits, test_y, T),
                 "brier_before": brier(test_logits, test_y), "brier_after": brier(test_logits, test_y, T)})
table = pd.DataFrame(rows)

# all test folds together (each image is a test image in exactly one round)
test_y = np.concatenate([data[k]["test"][1] for k in ROUNDS])
before = np.concatenate([data[k]["test"][0] for k in ROUNDS])
after = np.concatenate([data[k]["test"][0] / temperatures[k] for k in ROUNDS])

In [ ]:
# ============================== 6. RESULTS ==============================
def mean_std(values):
    return f"{np.mean(values):.3f} ± {np.std(values):.3f}" if len(values) > 1 else f"{values[0]:.3f}"

lines = [f"run = {RUN_NAME} | rounds {ROUNDS} | temperature fitted on validation, measured on test | {N_BINS} bins", "",
         "A. Per round (test fold)", table.round(4).to_string(index=False), "",
         f"B. Mean ± std over {len(ROUNDS)} round(s)"]
for column in table.columns[1:]:
    lines.append(f"   {column:<18} {mean_std(table[column].tolist())}")

# ---- C. what happens if we only accept confident answers (after calibration, all test folds together)
p = probabilities(after)
confidence, correct = p.max(1), p.argmax(1) == test_y
accept = pd.DataFrame([{"min_confidence": f"{t:.2f}","cells_kept_%": 100 * (confidence >= t).mean(),
                        "accuracy_on_kept_%": 100 * correct[confidence >= t].mean() if (confidence >= t).any() else np.nan,
                        "cells_sent_to_human": int((confidence < t).sum())} for t in THRESHOLDS])
lines += ["", "C. Accepting only confident answers (after calibration, all test folds together)",
          accept.round(1).to_string(index=False), "",
          "D. Calibration groups, all test folds together (n = images in the group)"]
for name, logits in [("before", before), ("after", after)]:
    lines += [f"   {name}: overall calibration error {ece(logits, test_y):.4f}",
              bins(logits, test_y).round(3).to_string(index=False), ""]

# ---- E. reliability chart
INK, INK_2, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e1e0d9"
SERIES = [("before", before, "#eb6834"), ("after", after, "#2a78d6")]
fig, (left, right) = plt.subplots(1, 2, figsize=(9.6, 4.2), facecolor=SURFACE)
left.plot([0, 1], [0, 1], color="#c3c2b7", linewidth=1, linestyle="--", label="perfect calibration")
for name, logits, colour in SERIES:
    group = bins(logits, test_y)
    group = group[group["n"] >= 30]                          # tiny groups are too noisy to draw
    left.plot(group["confidence"], group["accuracy"], color=colour, linewidth=2, marker="o", markersize=5,
              label=f"{name} (error {ece(logits, test_y):.3f})")
    right.hist(probabilities(logits).max(1), bins=N_BINS, range=(0, 1), histtype="step", linewidth=2, color=colour,
               label=name)
left.set_xlim(0.3, 1.0); left.set_ylim(0.3, 1.0)
left.set_xlabel("confidence the model states", color=INK_2, fontsize=9)
left.set_ylabel("share that is actually correct", color=INK_2, fontsize=9)
left.set_title("Reliability (test folds together)", loc="left", color=INK, fontsize=10)
right.set_xlim(0.3, 1.0); right.set_yscale("log")
right.set_xlabel("confidence the model states", color=INK_2, fontsize=9)
right.set_ylabel("number of cells (log scale)", color=INK_2, fontsize=9)
right.set_title("How often each confidence is used", loc="left", color=INK, fontsize=10)
for ax in (left, right):
    ax.set_facecolor(SURFACE); ax.grid(color=GRID, linewidth=0.8); ax.tick_params(colors=INK_2, labelsize=8)
    ax.legend(frameon=False, fontsize=8, labelcolor=INK)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
fig.tight_layout(); fig.savefig(CAL_DIR / "reliability.png", dpi=150); plt.show()

table.to_csv(CAL_DIR / "calibration_per_round.csv", index=False)
(CAL_DIR / "temperatures.json").write_text(json.dumps({"run": RUN_NAME, "fitted_on": "validation",
                                                        "temperature": {str(k): t for k, t in temperatures.items()}}, indent=2))
(CAL_DIR / "calibration_summary.txt").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))
print("Saved in", CAL_DIR, ": calibration_summary.txt, calibration_per_round.csv, temperatures.json, reliability.png")

## What to send back

Copy the **printed text of section 6** into the chat, and download the `calibration/` folder from
`My Drive/ANA_Pattern3/outputs/resnet18_adjusted/` into the project's `outputs/resnet18_adjusted/` folder.